<a href="https://colab.research.google.com/github/0xShaswatGit22/LoRA-LLaMA-Efficient-Fine-Tuning-of-LLMs-using-QLoRA/blob/main/llama3_1_sentiment_pyspark_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === INSTALLATIONS (Run once) ===
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets tqdm pandas

print("✅ Installation completed!")

In [ ]:
import pandas as pd
import re
from datasets import Dataset
from tqdm import tqdm
import csv

# Load your CSV
df = pd.read_csv(
    "/content/Amazon_Reviews.csv",
    engine='python',
    on_bad_lines='skip',
    encoding='utf-8',
    quoting=csv.QUOTE_MINIMAL
)

print(f"✅ Loaded {len(df):,} reviews")
print("Columns:", df.columns.tolist())

# Extract numeric rating
def extract_rating(text):
    match = re.search(r'(\d+)', str(text))
    return float(match.group(1)) if match else 3.0

df['rating_numeric'] = df['Rating'].apply(extract_rating)

# Sample for T4 (you can increase to 8000 later)
sample_size = 5000
df_sample = df.sample(n=sample_size, random_state=42).reset_index(drop=True)

def create_training_example(row):
    title = str(row['Review Title'])
    text = str(row['Review Text'])
    rating = row['rating_numeric']

    full_review = f"{title}. {text}"[:1450]

    sentiment = "Positive" if rating >= 4.0 else "Negative" if rating <= 2.0 else "Neutral"

    instruction = f"""Analyze this customer review and give structured sentiment output.

Review: {full_review}"""

    response = f"""Sentiment: {sentiment}
Rating: {rating}/5.0
Key Aspects: Quality, Service, Delivery, Value
Summary: Customer had a {sentiment.lower()} experience."""

    return {
        "text": f"<|begin_of_text|>User: {instruction}\n\nAssistant: {response}<|end_of_text|>"
    }

tqdm.pandas()
examples = df_sample.progress_apply(create_training_example, axis=1).tolist()

hf_dataset = Dataset.from_list(examples)

print(f"✅ Training dataset ready with {len(hf_dataset)} examples!")

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
    device_map = "auto",
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

print("✅ Model + LoRA loaded successfully!")
print(model.print_trainable_parameters())

In [ ]:
# === CLEAN + PROPER INSTALLATION (Run this first) ===
!pip uninstall -y unsloth unsloth_zoo

!pip install --upgrade unsloth_zoo
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

!pip install trl peft accelerate bitsandbytes datasets tqdm pandas

print("✅ Unsloth + Dependencies Installed Successfully!")

In [ ]:

import torch
from unsloth import FastLanguageModel
import pandas as pd
from datasets import Dataset
from tqdm import tqdm
import re
import csv

print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
# Load your CSV
df = pd.read_csv(
    "/content/Amazon_Reviews.csv",
    engine='python',
    on_bad_lines='skip',
    encoding='utf-8',
    quoting=csv.QUOTE_MINIMAL
)

print(f"✅ Loaded {len(df):,} reviews")
print("Columns:", df.columns.tolist())

# Extract rating
def extract_rating(text):
    match = re.search(r'(\d+)', str(text))
    return float(match.group(1)) if match else 3.0

df['rating_numeric'] = df['Rating'].apply(extract_rating)

# Sample
sample_size = 5000
df_sample = df.sample(n=sample_size, random_state=42).reset_index(drop=True)

def create_example(row):
    full_review = f"{str(row['Review Title'])}. {str(row['Review Text'])}"[:1450]
    rating = row['rating_numeric']
    sentiment = "Positive" if rating >= 4.0 else "Negative" if rating <= 2.0 else "Neutral"

    instruction = f"""Analyze this customer review and provide structured sentiment analysis.

Review: {full_review}"""

    response = f"""Sentiment: {sentiment}
Rating: {rating}/5.0
Key Aspects: Quality, Service, Delivery, Value
Summary: The customer had a {sentiment.lower()} experience."""

    return {
        "text": f"<|begin_of_text|>User: {instruction}\n\nAssistant: {response}<|end_of_text|>"
    }

tqdm.pandas(desc="Creating examples")
examples = df_sample.progress_apply(create_example, axis=1).tolist()

hf_dataset = Dataset.from_list(examples)
print(f"✅ Dataset ready with {len(hf_dataset)} examples!")

In [ ]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
    device_map = "auto",
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

print("✅ Model + LoRA Loaded!")
print(model.print_trainable_parameters())

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = hf_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        output_dir = "outputs",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        max_steps = 80,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        report_to = "none",
    ),
)

print("🚀 Training Started...")
trainer.train()
print("🎉 Training Finished!")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = hf_dataset,
    dataset_text_field = "text",
    max_seq_length = 1024,           # Reduced from 2048 → saves big memory
    args = TrainingArguments(
        output_dir = "outputs",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,     # Reduced
        max_steps = 60,                      # Reduced for faster testing
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        report_to = "none",

        # Extra memory saving options
        gradient_checkpointing = True,
        dataloader_num_workers = 0,
    ),
)

print("🚀 Starting lighter training (Lower memory usage)...")
trainer.train()
print("🎉 Training Finished!")

In [ ]:
# Save the trained LoRA adapters
model.save_pretrained("lora_sentiment_model")
tokenizer.save_pretrained("lora_sentiment_model")

print("✅ LoRA adapters saved successfully!")

In [ ]:
from unsloth import FastLanguageModel

# Merge LoRA weights
model = FastLanguageModel.for_inference(model)   # Switch to inference mode

print("✅ Model ready for inference!")

In [ ]:
def analyze_review(review_text):
    prompt = f"""Analyze this customer review and provide structured sentiment analysis.

Review: {review_text}"""

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        use_cache=True
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("Assistant:")[-1].strip()   # Extract only assistant part

# Test with some examples
test_reviews = [
    "The product is amazing! It works perfectly and delivery was super fast.",
    "Worst purchase ever. The item broke in 2 days and customer service is terrible.",
    "Average product. It's okay for the price but nothing special."
]

for review in test_reviews:
    print("="*80)
    print("Review:", review)
    print("Model Output:")
    print(analyze_review(review))
    print("="*80)

In [ ]:
# === SETUP PYSPARK ===
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install pyspark==3.5.1

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType

spark = SparkSession.builder \
    .appName("LlamaSentimentAnalysis") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("✅ PySpark Session Created!")

In [ ]:
def analyze_review(review_text):
    try:
        prompt = f"""Analyze this customer review and provide structured sentiment analysis.

Review: {review_text}"""

        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
            use_cache=True
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Extract only the Assistant's response
        return response.split("Assistant:")[-1].strip()

    except Exception as e:
        return f"Error: {str(e)}"

print("✅ analyze_review function defined successfully!")

In [ ]:
# Register your existing analyze_review function as Spark UDF
sentiment_udf = udf(analyze_review, StringType())

print("✅ Spark UDF created using your fine-tuned Llama model!")

In [ ]:
# Load your Amazon Reviews CSV using PySpark
df_spark = spark.read.option("header", True) \
                     .option("multiLine", True) \
                     .option("quote", "\"") \
                     .option("escape", "\"") \
                     .csv("/content/Amazon_Reviews.csv")

# Basic cleaning
df_spark = df_spark.withColumn("clean_review",
                               col("Review Text").cast("string")) \
                   .na.drop(subset=["clean_review"])

print(f"Total Reviews Loaded: {df_spark.count():,}")

In [ ]:
# Apply your fine-tuned model on the entire dataset
result_df = df_spark.withColumn("Llama_Sentiment", sentiment_udf(col("clean_review")))

# Show sample results
print("=== Sample Predictions ===")
result_df.select(
    col("Review Title"),
    col("Review Text").substr(1, 150).alias("Review_Sample"),
    "Llama_Sentiment"
).show(8, truncate=False)

In [ ]:
# Sentiment Distribution
insights = result_df.groupBy("Llama_Sentiment").count().orderBy("count", ascending=False)
print("=== Sentiment Distribution ===")
insights.show(truncate=False)

# Top Positive & Negative Reviews (Optional)
result_df.filter(col("Llama_Sentiment").contains("Positive")).select("Review Title", "Review Text").show(5, truncate=100)
result_df.filter(col("Llama_Sentiment").contains("Negative")).select("Review Title", "Review Text").show(5, truncate=100)

In [ ]:
# Save as Parquet (Industry standard for big data)
result_df.write.mode("overwrite").parquet("/content/sentiment_analysis_results")

print("✅ Results saved successfully as Parquet!")